# Luna NNUE - bullet on Colab (CUDA), prepared, NOT yet run

Written 2026-09-25 without Colab access: **nothing in this notebook has been executed on Colab.** What was verified elsewhere
(CPU backend, PC and Oracle): the training example, checkpoint save/resume (bit-identical on the CPU backend, single thread),
the converter with its round-trip gate. Not verified: the CUDA build of the fork at commit `6a4f4fb`, the Drive paths, the
streaming download inside a session. Runtime: GPU (T4). Order: run the cells top to bottom; after a disconnect run 1-3 again
and then cell 5 - it resumes from the newest checkpoint on Drive by itself.

In [ ]:
# 1. Drive + configuration
from google.colab import drive
drive.mount('/content/drive')
import os
RUN = '/content/drive/MyDrive/luna_nnue/run_A'          # checkpoints + logs live here (survive the session)
os.makedirs(RUN, exist_ok=True)
POSITIONS   = 250_000_000     # distinct positions to stream (32 bytes each); Colab local disk decides how many fit
SB, BPS     = 60, 1017        # recipe: 60 superbatches x 1017 batches x 4096 = 250 M samples seen (same as the Oracle runs)
SAVE_EVERY  = 5               # superbatches between checkpoints (a checkpoint is ~3x the net: weights + AdamW moments)
DATA = '/content/data.bin'
!nvidia-smi -L; !df -h /content | tail -1

In [ ]:
# 2. Toolchain and bullet (Petrel's fork: the one the recipe and the example were written against)
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal >/dev/null
import os; os.environ['PATH'] += ':/root/.cargo/bin'
!rm -rf /content/bullet && git clone -q https://github.com/AleksPeshkov/bullet /content/bullet && cd /content/bullet && git checkout -q 6a4f4fb && git log -1 --oneline
!which nvcc || echo "no nvcc: the CUDA backend needs the toolkit (Colab GPU images normally have it)"

In [ ]:
%%writefile /content/bullet/examples/luna_pilot.rs
// Luna phase-2 pilot: (768hm -> 1024)x2 -> 1 SCReLU, no buckets, float export (quantised afterwards by our own script
// so that every gate is ours). Env: LP_DATA (comma list of .bin), LP_SB (superbatches), LP_BPS (batches/superbatch),
// LP_BATCH, LP_THREADS, LP_OUT.
use bullet_lib::{
    game::inputs::Chess768hm,
    nn::optimiser::AdamW,
    trainer::{
        save::SavedFormat,
        schedule::{TrainingSchedule, TrainingSteps, lr, wdl},
        settings::LocalSettings,
    },
    value::{ValueTrainerBuilder, loader::DirectSequentialDataLoader},
};

fn env<T: std::str::FromStr>(k: &str, d: T) -> T {
    std::env::var(k).ok().and_then(|v| v.parse().ok()).unwrap_or(d)
}

fn main() {
    const HIDDEN: usize = 1024;
    let threads: usize = env("LP_THREADS", 4);
    let sb: usize = env("LP_SB", 10);
    let bps: usize = env("LP_BPS", 100);
    let batch: usize = env("LP_BATCH", 4096);
    let start_sb: usize = env("LP_START", 1); // resume: first superbatch still to run (checkpoint N done -> N+1)
    let save: usize = env("LP_SAVE", sb);
    let resume: String = env("LP_RESUME", String::new());
    let out: String = env("LP_OUT", "checkpoints".to_string());
    let data: Vec<String> = std::env::var("LP_DATA").expect("LP_DATA").split(',').map(String::from).collect();
    let data: Vec<&str> = data.iter().map(|s| s.as_str()).collect();

    let mut trainer = ValueTrainerBuilder::default()
        .use_threads(threads)
        .optimiser(AdamW)
        .loss_fn(|output, target| output.sigmoid().squared_error(target))
        // raw f32 tensors: the quantisation and every gate are done by the conversion script
        .save_format(&[SavedFormat::id("l0w"), SavedFormat::id("l0b"), SavedFormat::id("l1w"), SavedFormat::id("l1b")])
        .inputs(Chess768hm)
        .dual_perspective()
        .build(|builder, stm, ntm| {
            let l0 = builder.new_affine("l0", 768, HIDDEN);
            let l1 = builder.new_affine("l1", 2 * HIDDEN, 1);
            let a = l0.forward(stm).screlu();
            let b = l0.forward(ntm).screlu();
            l1.forward(a.concat(b))
        });

    if !resume.is_empty() {
        trainer.load_from_checkpoint(&resume); // weights + AdamW moments; the loader skips to batch (start_sb-1)*bps
    }
    let loader = DirectSequentialDataLoader::new(&data);
    let schedule = TrainingSchedule {
        net_id: "luna_pilot".to_string(),
        eval_scale: 400.0,
        steps: TrainingSteps { batch_size: batch, batches_per_superbatch: bps, start_superbatch: start_sb, end_superbatch: sb },
        // phase-3 recipe (Petrel): WDL fraction ramps 0.0 -> 0.1 (bullet's convention), lr cosine 4e-4 -> peak/40
        wdl_scheduler: wdl::CosineDecayWDL { start: 0.0, end: 0.1, final_superbatch: sb },
        lr_scheduler: lr::CosineDecayLR { initial_lr: 0.0004, final_lr: 0.0004 / 40.0, final_superbatch: sb },
        save_rate: save,
    };
    let settings = LocalSettings { threads: 2, test_set: None, output_directory: &out, batch_queue_size: 32 };
    trainer.run(&schedule, &settings, &loader);
}

In [ ]:
# 3. build (CUDA is bullet_lib's default feature). Add the example entry, then build.
!cd /content/bullet && printf '
[[example]]
name = "luna_pilot"
path = "../../examples/luna_pilot.rs"
' >> crates/bullet_lib/Cargo.toml
!cd /content/bullet && cargo build -r --example luna_pilot 2>&1 | tail -3
!ls -la /content/bullet/target/release/examples/luna_pilot

In [ ]:
# 4. data: stream the first POSITIONS records of S2 iter-1 straight into the local disk (no compressed copy kept).
# ~1.94 bytes of decompressed data per compressed byte -> range = POSITIONS*32/1.9 + margin. NOTE: bullet does NOT shuffle;
# the files are not shuffled either (adjacent records are from the same game: measured 2026-09-25).
U = 'https://huggingface.co/datasets/linrock/bullet-training-data/resolve/main/S2/test77nov-unfilt-test79-maraprmay-v6-dd.skip-see-ge0.wdl-pdist.iter-1.bullet.bin.zst'
rng = int(POSITIONS*32/1.9) + 200_000_000
!apt-get -qq install -y zstd >/dev/null
if not os.path.exists(DATA) or os.path.getsize(DATA) != POSITIONS*32:
    !curl -sL -r 0-{rng} "{U}" | zstd -dc 2>/dev/null | head -c {POSITIONS*32} > {DATA}
print(os.path.getsize(DATA)/32, 'positions')

In [ ]:
# 5. train, resuming from the newest checkpoint on Drive if there is one
import re, glob, subprocess
cks = sorted(glob.glob(RUN + '/luna_pilot-*'), key=lambda p: int(p.rsplit('-',1)[1]))
env = dict(os.environ, LP_DATA=DATA, LP_SB=str(SB), LP_BPS=str(BPS), LP_SAVE=str(SAVE_EVERY), LP_OUT=RUN, LP_THREADS='1')
if cks:
    last = int(cks[-1].rsplit('-',1)[1]); env.update(LP_RESUME=cks[-1], LP_START=str(last+1)); print('resuming from', cks[-1])
else:
    print('fresh start')
subprocess.run(['/content/bullet/target/release/examples/luna_pilot'], env=env, check=True)

In [ ]:
# 6. after training: convert + round-trip (needs a Luna exe with the `eval` command: build the v3.1.6 tag)
!cd /content && rm -rf luna_src && git clone -q --branch v3.1.6 https://github.com/Spunc595/Luna-Chess-Engine luna_src && cd luna_src && cargo build -r 2>&1 | tail -1
!pip -q install python-chess numpy

In [ ]:
%%writefile /content/bullet_luna.py
"""bullet (Chess768hm, no buckets, (768->1024)x2->1) checkpoint -> Luna net.bin, plus an independent reference inference.

convert(): the conversion, including the mirror-direction permutation (`mirror_fix`). The reference (`Reference`) is written
from bullet's Chess768hm definition only and never reads Luna's code or the converted file: it is the independent side of
the round-trip. `mirror_fix=False` exists ONLY so the test can prove that the round-trip fails without the permutation.
"""
import numpy as np

QA, QB, HIDDEN, NB = 255, 64, 1024, 4
I16MAX, I16MIN = 32767, -32768
RAW_FLOATS = 768 * HIDDEN + HIDDEN + 2 * HIDDEN + 1


class GateError(Exception):
    pass


def convert(raw_f32, mirror_fix=True):
    """raw_f32: flat f32 array (l0w[768][1024] feature-major, l0b, l1w[2][1024] = stm|ntm, l1b). Returns net.bin bytes,
    or raises GateError (nothing is clamped silently)."""
    raw = np.asarray(raw_f32, dtype="<f4")
    assert raw.size == RAW_FLOATS, raw.size
    l0w = raw[:768 * HIDDEN].reshape(768, HIDDEN)
    if mirror_fix:
        # bullet's Chess768hm flips (sq ^ 7) when the king is on files a-d (`our_ksq & 4 == 0`) -> king on e-h; Luna flips
        # when the king is on e-h -> king on a-d. They differ by one file flip: Luna row (pt, sq) = bullet row (pt, sq ^ 7).
        l0w = l0w.reshape(12, 64, HIDDEN)[:, np.arange(64) ^ 7, :].reshape(768, HIDDEN)
    l0b = raw[768 * HIDDEN:769 * HIDDEN]
    l1w = raw[769 * HIDDEN:771 * HIDDEN].reshape(2, HIDDEN)
    l1b = raw[-1:]
    q = lambda a, s: np.round(a.astype(np.float64) * s)
    fw, fb, ow, ob = q(l0w, QA), q(l0b, QA), q(l1w, QB), q(l1b, QA * QB)
    problems = []
    for n, a in (("feature_weights", fw), ("feature_bias", fb), ("output_weights", ow), ("output_bias", ob)):
        c = int(((a >= I16MAX) | (a <= I16MIN)).sum())
        if c:
            problems.append(f"{n}: {c} saturated")
    if QA * int(np.abs(ow).max()) > I16MAX:  # SIMD gate
        problems.append("SIMD gate: 255 * max|output weight| > 32767")
    w = fw.astype(np.int64)
    up = fb.astype(np.int64) + np.sort(w, axis=0)[-32:].clip(min=0).sum(axis=0)
    lo = fb.astype(np.int64) + np.sort(w, axis=0)[:32].clip(max=0).sum(axis=0)
    if up.max() > I16MAX or lo.min() < I16MIN:
        problems.append(f"accumulator bound: max {up.max()} min {lo.min()}")
    if problems:
        raise GateError("; ".join(problems))
    out = (np.tile(fw.astype("<i2"), (NB, 1)).tobytes() + fb.astype("<i2").tobytes() + ow.astype("<i2").tobytes()
           + ob.astype("<i2").tobytes() + bytes(62))
    assert len(out) == 6_297_664
    return out


_PT = {c: i for i, c in enumerate("pnbrqk")}


def _features(fen):
    bd, stm = fen.split()[0], fen.split()[1]
    pcs = []  # (0 = side to move's piece / 1 = opponent's, piece type, square in the stm-relative frame)
    for r, row in enumerate(bd.split("/")):
        f = 0
        for ch in row:
            if ch.isdigit():
                f += int(ch)
                continue
            sq = (7 - r) * 8 + f
            if stm == "b":
                sq ^= 56
            pcs.append((0 if ch.isupper() == (stm == "w") else 1, _PT[ch.lower()], sq))
            f += 1
    ks = [s for c, p, s in pcs if p == 5 and c == 0][0]
    kn = [s for c, p, s in pcs if p == 5 and c == 1][0]
    hs, hn = (0 if ks & 4 else 7), (0 if kn & 4 else 7)   # bullet's Chess768hm, verbatim
    return ([[0, 384][c] + 64 * p + (s ^ hs) for c, p, s in pcs],
            [[384, 0][c] + 64 * p + (s ^ hn ^ 56) for c, p, s in pcs])


class Reference:
    def __init__(self, raw_f32):
        raw = np.asarray(raw_f32, dtype="<f4")
        q = lambda a, s: np.round(a.astype(np.float64) * s).astype(np.int64)
        self.W = q(raw[:768 * HIDDEN].reshape(768, HIDDEN), QA)
        self.B = q(raw[768 * HIDDEN:769 * HIDDEN], QA)
        self.OW = q(raw[769 * HIDDEN:771 * HIDDEN].reshape(2, HIDDEN), QB)
        self.OB = int(q(raw[-1:], QA * QB)[0])

    def eval(self, fen):
        s = 0
        for k, idx in enumerate(_features(fen)):
            c = np.clip(self.B + self.W[idx].sum(axis=0), 0, QA)
            s += int((c * c * self.OW[k]).sum())
        out = int(s / QA) + self.OB   # truncation toward zero, like Rust's i32 division
        return max(-15000, min(15000, int(out * 400 / (QA * QB))))


def luna_eval(exe, net_path, fens):
    """Luna's own static `eval` (no search) for each FEN; exe copied into a private folder with the net as luna.nnue."""
    import os
    import shutil
    import subprocess
    import tempfile
    tmp = tempfile.mkdtemp(prefix="luna_rt_")
    try:
        e = os.path.join(tmp, os.path.basename(exe))
        shutil.copy2(exe, e)
        shutil.copy2(net_path, os.path.join(tmp, "luna.nnue"))
        p = subprocess.Popen([e], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1, cwd=tmp,
                             encoding="utf-8", errors="replace")

        def rd(pref):
            while True:
                line = p.stdout.readline()
                if not line:
                    raise RuntimeError("engine ended early")
                if line.startswith(pref):
                    return line

        p.stdin.write("uci\n")
        rd("uciok")
        out = []
        for fen in fens:
            p.stdin.write(f"position fen {fen}\neval\n")
            out.append(int(float(rd("Evaluation:").split()[1])))
        p.stdin.write("quit\n")
        p.wait(timeout=10)
        return out
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def sample_positions(n, seed=7):
    """Seeded random legal positions (both colours to move, kings on every half of the board)."""
    import random
    import chess
    rng = random.Random(seed)
    out = []
    while len(out) < n:
        b = chess.Board()
        for _ in range(rng.randint(6, 90)):
            m = list(b.legal_moves)
            if not m:
                break
            b.push(rng.choice(m))
        if not b.is_game_over() and not b.is_check():
            out.append(b.fen())
    return out


class RoundTripError(Exception):
    pass


def convert_verified(raw_f32, out_path, exe, n=2000, seed=11):
    """The only sanctioned way to produce a net.bin: convert (all gates), then evaluate `n` positions with the independent
    reference and with the engine `exe` loading the converted bytes as luna.nnue, and write `out_path` ONLY if there is not
    one single difference. There is no path from a checkpoint to a file that skips the round-trip."""
    import os
    import tempfile
    data = convert(raw_f32)
    fens = sample_positions(n, seed)
    ref = Reference(raw_f32)
    expected = [ref.eval(f) for f in fens]
    fd, tmp = tempfile.mkstemp(suffix=".nnue")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
        got = luna_eval(exe, tmp, fens)
    finally:
        os.unlink(tmp)
    diffs = [i for i, (a, b) in enumerate(zip(expected, got)) if a != b]
    if len(got) != len(fens) or diffs:
        raise RoundTripError(f"round-trip FAILED: {len(diffs)} of {len(fens)} positions differ (first {diffs[:5]}); {out_path} not written")
    with open(out_path, "wb") as f:
        f.write(data)
    return len(fens)

In [ ]:
%%writefile /content/convert_bullet_to_luna.py
"""Usage: convert_bullet_to_luna.py raw.bin net.bin --exe luna.exe   (or LUNA_EXE=...)

Gates AND the round-trip (2000 positions, independent reference vs the engine) run inside bullet_luna.convert_verified:
the file is written only if every gate passes and there is zero difference. Exit code 1 otherwise, no file created."""
import argparse
import os
import sys

import numpy as np

import bullet_luna as bl

ap = argparse.ArgumentParser()
ap.add_argument("raw")
ap.add_argument("out")
ap.add_argument("--exe", default=os.environ.get("LUNA_EXE"))
ap.add_argument("--n", type=int, default=2000)
a = ap.parse_args()
if not a.exe or not os.path.exists(a.exe):
    sys.exit("REFUSED: an engine executable is required for the round-trip (--exe or LUNA_EXE)")
try:
    n = bl.convert_verified(np.fromfile(a.raw, dtype="<f4"), a.out, a.exe, a.n)
except (bl.GateError, bl.RoundTripError) as e:
    print("REFUSED:", e)
    sys.exit(1)
print(f"round-trip OK ({n} positions, 0 differences); wrote {a.out}")

In [ ]:
final = sorted(glob.glob(RUN + '/luna_pilot-*'), key=lambda p: int(p.rsplit('-',1)[1]))[-1]
!cd /content && python convert_bullet_to_luna.py {final}/raw.bin {RUN}/net.nnue --exe /content/luna_src/target/release/luna
# refuses (exit 1, no file) on any gate or any round-trip difference. Then measure with pipeline/measure/measure_sf18_evalset.py
# (CSV on Drive) exactly as for the PC/Oracle runs.